In [2]:
# Cell 1: Import Libraries

import pandas as pd
import numpy as np
import ast
import re
from collections import Counter, defaultdict

In [3]:
# Cell 2: Load Job Market Dataset

df_jobs = pd.read_csv("job_data.csv")

print("Dataset loaded successfully!")
print("Rows:", df_jobs.shape[0])
print("Columns:", df_jobs.shape[1])

Dataset loaded successfully!
Rows: 200
Columns: 8


In [4]:
# Dataset shape

print("Shape:", df_jobs.shape)

print("\nColumns:")
print(df_jobs.columns.tolist())

print("\nFirst 5 rows:")
display(df_jobs.head())

Shape: (200, 8)

Columns:
['Job ID', 'Job Title', 'Industry', 'Required Education', 'Required Degree Field', 'Years of Experience', 'Hard Skills', 'Soft Skills']

First 5 rows:


,Job ID,Job Title,Industry,Required Education,Required Degree Field,Years of Experience,Hard Skills,Soft Skills
0,1,Software Engineer,Technology,Bachelor,Computing,2,"[""Python"", ""Java"", ""C++"", ""Git"", ""Version Cont...","['Problem-Solving', 'Teamwork']"
1,2,Frontend Developer,Technology,Bachelor,Computing,1,"[""HTML"", ""CSS"", ""JavaScript"", ""React"", ""TypeSc...","['Creativity', 'Attention to Detail']"
2,3,Backend Developer,Technology,Bachelor,Computing,2,"[""Java"", ""Python"", ""Node.js"", ""Spring Boot"", ""...","['Logical Thinking', 'Collaboration']"
3,4,Full Stack Developer,Technology,Bachelor,Computing,3,"[""JavaScript"", ""React"", ""Node.js"", ""MongoDB"", ...","['Adaptability', 'Communication']"
4,5,DevOps Engineer,Technology,Bachelor,Computing,3,"[""Docker"", ""Kubernetes"", ""CI/CD"", ""AWS"", ""Terr...","['Problem-Solving', 'Teamwork']"


In [5]:
# Dataset information

df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Job ID                 200 non-null    int64 
 1   Job Title              200 non-null    object
 2   Industry               200 non-null    object
 3   Required Education     178 non-null    object
 4   Required Degree Field  193 non-null    object
 5   Years of Experience    200 non-null    int64 
 6   Hard Skills            200 non-null    object
 7   Soft Skills            200 non-null    object
dtypes: int64(2), object(6)
memory usage: 12.6+ KB


In [6]:
# Check missing values

missing_values = df_jobs.isnull().sum()

print("Missing values:")
print(missing_values)

Missing values:
Job ID                    0
Job Title                 0
Industry                  0
Required Education       22
Required Degree Field     7
Years of Experience       0
Hard Skills               0
Soft Skills               0
dtype: int64


In [7]:
# Check duplicate rows

duplicate_rows = df_jobs.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

# Check duplicate Job IDs

duplicate_ids = df_jobs["Job ID"].duplicated().sum()

print("Duplicate Job IDs:", duplicate_ids)

Duplicate rows: 0
Duplicate Job IDs: 0


In [8]:
# Standardize column names

df_jobs.columns = (
    df_jobs.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df_jobs.columns.tolist())

['job_id', 'job_title', 'industry', 'required_education', 'required_degree_field', 'years_of_experience', 'hard_skills', 'soft_skills']


In [9]:
# Clean text columns

text_columns = [
    "job_title",
    "industry",
    "required_education",
    "required_degree_field"
]

for col in text_columns:
    df_jobs[col] = (
        df_jobs[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Text cleaning completed.")

Text cleaning completed.


In [10]:
# Convert years of experience to numeric

df_jobs["years_of_experience"] = pd.to_numeric(
    df_jobs["years_of_experience"],
    errors="coerce"
)

print(df_jobs["years_of_experience"].describe())

count    200.000000
mean       2.705000
std        1.176818
min        0.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        5.000000
Name: years_of_experience, dtype: float64


In [11]:
def parse_skill_list(value):
    if pd.isna(value):
        return []
    
    try:
        skills = ast.literal_eval(value)
        
        if isinstance(skills, list):
            return [
                str(skill).strip()
                for skill in skills
                if str(skill).strip()
            ]
        
    except (ValueError, SyntaxError):
        pass
    
    return []


df_jobs["hard_skills"] = df_jobs["hard_skills"].apply(parse_skill_list)
df_jobs["soft_skills"] = df_jobs["soft_skills"].apply(parse_skill_list)

print("Skill lists parsed successfully.")

display(
    df_jobs[
        ["job_title", "hard_skills", "soft_skills"]
    ].head()
)

Skill lists parsed successfully.


,job_title,hard_skills,soft_skills
0,Software Engineer,"[Python, Java, C++, Git, Version Control, Algo...","[Problem-Solving, Teamwork]"
1,Frontend Developer,"[HTML, CSS, JavaScript, React, TypeScript, Res...","[Creativity, Attention to Detail]"
2,Backend Developer,"[Java, Python, Node.js, Spring Boot, REST APIs...","[Logical Thinking, Collaboration]"
3,Full Stack Developer,"[JavaScript, React, Node.js, MongoDB, Express....","[Adaptability, Communication]"
4,DevOps Engineer,"[Docker, Kubernetes, CI/CD, AWS, Terraform, Je...","[Problem-Solving, Teamwork]"


In [12]:
def normalize_skill(skill):
    skill = str(skill).strip().lower()
    skill = re.sub(r"\s+", " ", skill)
    return skill


df_jobs["hard_skills_normalized"] = df_jobs["hard_skills"].apply(
    lambda skills: [normalize_skill(skill) for skill in skills]
)

df_jobs["soft_skills_normalized"] = df_jobs["soft_skills"].apply(
    lambda skills: [normalize_skill(skill) for skill in skills]
)

print("Skill normalization completed.")

Skill normalization completed.


In [13]:
### Algorithm 1 — Keyword-Family Inverted Index

#The project documentation specifies keyword-family inverted index filtering for job-role matching.


# Keyword families for job-role filtering

keyword_families = {
    "software_engineer": [
        "software engineer",
        "software developer",
        "application developer",
        "backend engineer",
        "backend developer",
        "full stack developer",
        "full-stack developer"
    ],
    
    "machine_learning_engineer": [
        "machine learning engineer",
        "ml engineer",
        "machine learning developer",
        "ml developer",
        "ai engineer",
        "artificial intelligence engineer"
    ],
    
    "data_scientist": [
        "data scientist",
        "data science",
        "data analyst",
        "analytics"
    ],
    
    "frontend_developer": [
        "frontend developer",
        "front end developer",
        "ui developer",
        "web developer"
    ],
    
    "backend_developer": [
        "backend developer",
        "back end developer",
        "backend engineer"
    ]
}

print("Keyword families created:")
for family, keywords in keyword_families.items():
    print(f"{family}: {keywords}")

Keyword families created:
software_engineer: ['software engineer', 'software developer', 'application developer', 'backend engineer', 'backend developer', 'full stack developer', 'full-stack developer']
machine_learning_engineer: ['machine learning engineer', 'ml engineer', 'machine learning developer', 'ml developer', 'ai engineer', 'artificial intelligence engineer']
data_scientist: ['data scientist', 'data science', 'data analyst', 'analytics']
frontend_developer: ['frontend developer', 'front end developer', 'ui developer', 'web developer']
backend_developer: ['backend developer', 'back end developer', 'backend engineer']


In [14]:
# Build keyword-family inverted index

inverted_index = defaultdict(set)

for idx, title in df_jobs["job_title"].items():
    
    title_lower = str(title).lower()
    
    for family, keywords in keyword_families.items():
        
        for keyword in keywords:
            
            if keyword in title_lower:
                inverted_index[family].add(idx)

print("Inverted index created.")

for family, job_indices in inverted_index.items():
    print(
        family,
        "->",
        len(job_indices),
        "matching jobs"
    )

Inverted index created.
software_engineer -> 3 matching jobs
frontend_developer -> 2 matching jobs
backend_developer -> 1 matching jobs
machine_learning_engineer -> 1 matching jobs
data_scientist -> 2 matching jobs


In [15]:
def search_jobs_by_role(role):
    
    role = role.lower().strip()
    
    if role not in inverted_index:
        return pd.DataFrame()
    
    indices = list(inverted_index[role])
    
    return df_jobs.loc[
        indices,
        [
            "job_id",
            "job_title",
            "industry",
            "required_education",
            "required_degree_field",
            "years_of_experience"
        ]
    ].reset_index(drop=True)


# Example search
software_jobs = search_jobs_by_role("software_engineer")

print("Software Engineer Jobs:")
display(software_jobs.head(10))

Software Engineer Jobs:


,job_id,job_title,industry,required_education,required_degree_field,years_of_experience
0,1,Software Engineer,Technology,Bachelor,Computing,2
1,3,Backend Developer,Technology,Bachelor,Computing,2
2,4,Full Stack Developer,Technology,Bachelor,Computing,3


In [16]:
### Algorithm 2 — TF Skill Distribution

##The project specifies TF skill distribution to determine skill demand in matching job postings.

#Combine Skills

# Combine hard and soft skills for each job

df_jobs["all_skills"] = (
    df_jobs["hard_skills_normalized"] +
    df_jobs["soft_skills_normalized"]
)

display(
    df_jobs[
        ["job_title", "all_skills"]
    ].head()
)
    

,job_title,all_skills
0,Software Engineer,"[python, java, c++, git, version control, algo..."
1,Frontend Developer,"[html, css, javascript, react, typescript, res..."
2,Backend Developer,"[java, python, node.js, spring boot, rest apis..."
3,Full Stack Developer,"[javascript, react, node.js, mongodb, express...."
4,DevOps Engineer,"[docker, kubernetes, ci/cd, aws, terraform, je..."


In [17]:

# Count skill occurrences across all job postings

skill_counter = Counter()

for skills in df_jobs["all_skills"]:
    for skill in skills:
        skill_counter[skill] += 1


skill_frequency = pd.DataFrame(
    skill_counter.items(),
    columns=["Skill", "Job_Count"]
)

skill_frequency = skill_frequency.sort_values(
    by="Job_Count",
    ascending=False
).reset_index(drop=True)

print("Most frequently requested skills:")

display(skill_frequency.head(20))

Most frequently requested skills:


,Skill,Job_Count
0,communication,78
1,attention to detail,51
2,problem-solving,50
3,creativity,37
4,leadership,29
5,collaboration,25
6,teamwork,23
7,analytical thinking,22
8,patience,18
9,critical thinking,16


In [18]:
def calculate_skill_demand(job_indices):
    
    if len(job_indices) == 0:
        return pd.DataFrame(
            columns=[
                "Skill",
                "Job_Count",
                "Demand_Ratio"
            ]
        )
    
    counter = Counter()
    
    for idx in job_indices:
        
        skills = set(
            df_jobs.loc[idx, "all_skills"]
        )
        
        for skill in skills:
            counter[skill] += 1
    
    total_jobs = len(job_indices)
    
    result = pd.DataFrame(
        counter.items(),
        columns=["Skill", "Job_Count"]
    )
    
    result["Demand_Ratio"] = (
        result["Job_Count"] / total_jobs
    )
    
    result = result.sort_values(
        by="Demand_Ratio",
        ascending=False
    ).reset_index(drop=True)
    
    return result

In [19]:
# Skill demand for Software Engineer jobs

software_indices = list(
    inverted_index["software_engineer"]
)

software_skill_demand = calculate_skill_demand(
    software_indices
)

display(
    software_skill_demand.head(20)
)

,Skill,Job_Count,Demand_Ratio
0,rest apis,3,1.000000
1,ci/cd,3,1.000000
2,python,2,0.666667
3,git,2,0.666667
4,agile,2,0.666667
5,java,2,0.666667
6,node.js,2,0.666667
7,unit testing,1,0.333333
8,algorithms,1,0.333333
9,oop,1,0.333333


In [20]:
# Skill demand for Machine Learning Engineer jobs

ml_indices = list(
    inverted_index["machine_learning_engineer"]
)

ml_skill_demand = calculate_skill_demand(
    ml_indices
)

display(
    ml_skill_demand.head(20)
)

,Skill,Job_Count,Demand_Ratio
0,scikit-learn,1,1.0
1,pytorch,1,1.0
2,"cloud platforms (aws, azure, gcp)",1,1.0
3,kubernetes,1,1.0
4,numpy,1,1.0
5,data modeling,1,1.0
6,python,1,1.0
7,analytical thinking,1,1.0
8,creativity,1,1.0
9,algorithm optimization,1,1.0


In [21]:
# Check whether salary information exists

salary_columns = [
    col for col in df_jobs.columns
    if "salary" in col.lower()
    or "compensation" in col.lower()
    or "pay" in col.lower()
    or "currency" in col.lower()
]

print("Salary-related columns found:")
print(salary_columns)

if len(salary_columns) == 0:
    print(
        "\nNo salary/currency column exists in job_data.csv."
    )
    print(
        "Salary normalization will be performed using "
        "the second Job Market dataset."
    )

Salary-related columns found:
[]

No salary/currency column exists in job_data.csv.
Salary normalization will be performed using the second Job Market dataset.


In [22]:
# Create final processed records

processed_jobs = []

for _, row in df_jobs.iterrows():
    
    record = {
        "job_id": row["job_id"],
        "job_title": row["job_title"],
        "industry": row["industry"],
        "required_education": row["required_education"],
        "required_degree_field": row["required_degree_field"],
        "years_of_experience": (
            None
            if pd.isna(row["years_of_experience"])
            else float(row["years_of_experience"])
        ),
        "hard_skills": row["hard_skills_normalized"],
        "soft_skills": row["soft_skills_normalized"],
        "all_skills": row["all_skills"]
    }
    
    processed_jobs.append(record)


print("Processed records:", len(processed_jobs))

Processed records: 200


In [23]:
# Save processed jobs as JSON

with open("jobs.json", "w", encoding="utf-8") as f:
    json.dump(
        processed_jobs,
        f,
        indent=2,
        ensure_ascii=False
    )

print("jobs.json saved successfully!")

jobs.json saved successfully!


In [24]:
# Save processed dataset as CSV

df_jobs.to_csv(
    "jobs_processed.csv",
    index=False
)

print("jobs_processed.csv saved successfully!")


jobs_processed.csv saved successfully!


In [25]:
# Final validation

print("========== FINAL VALIDATION ==========")

print("Number of job records:", len(processed_jobs))

print(
    "Unique Job IDs:",
    df_jobs["job_id"].nunique()
)

print(
    "Missing Job Titles:",
    df_jobs["job_title"].isna().sum()
)

print(
    "Missing Hard Skill Lists:",
    df_jobs["hard_skills"].apply(len).eq(0).sum()
)

print(
    "Missing Soft Skill Lists:",
    df_jobs["soft_skills"].apply(len).eq(0).sum()
)

print("\nFiles created:")
print("1. jobs.json")
print("2. jobs_processed.csv")

print("\nJob Market preprocessing completed!")

========== FINAL VALIDATION ==========
Number of job records: 200
Unique Job IDs: 200
Missing Job Titles: 0
Missing Hard Skill Lists: 0
Missing Soft Skill Lists: 0

Files created:
1. jobs.json
2. jobs_processed.csv

Job Market preprocessing completed!


In [26]:
# Cell 25: Load LinkedIn Job Postings Dataset

df_linkedin = pd.read_csv("linkedin_job_postings_dataset.csv")

print("Dataset loaded successfully!")
print("Rows:", df_linkedin.shape[0])
print("Columns:", df_linkedin.shape[1])

display(df_linkedin.head())

Dataset loaded successfully!
Rows: 500
Columns: 12


,job_id,job_title,company,location,employment_type,experience_level,industry,skills_required,salary_min_usd,salary_max_usd,remote_allowed,posted_date
0,1,Product Manager,Meta,"Austin, USA",Full-time,Entry level,Education,"Python, Node.js, SQL, JavaScript, Power BI, Ma...",53278,99326,True,25-04-2023
1,2,Product Manager,Oracle,"London, UK",Full-time,Mid level,E-commerce,"AWS, Java, JavaScript, Azure, SQL",76062,141049,True,28-07-2024
2,3,Machine Learning Engineer,Salesforce,"London, UK",Full-time,Director,Education,"Node.js, Docker, SQL",62676,119728,False,25-01-2024
3,4,Data Analyst,StartupX,"Seattle, USA",Contract,Entry level,Finance,"Docker, AWS, Kubernetes",75203,94319,False,06-01-2024
4,5,Product Manager,StartupX,"London, UK",Full-time,Mid level,Education,"Java, Kubernetes, Docker, AWS",98520,155086,True,16-06-2023


In [27]:
# Cell 26: Dataset Inspection

print("Shape:", df_linkedin.shape)

print("\nColumns:")
print(df_linkedin.columns.tolist())

print("\nDataset information:")
df_linkedin.info()


Shape: (500, 12)

Columns:
['job_id', 'job_title', 'company', 'location', 'employment_type', 'experience_level', 'industry', 'skills_required', 'salary_min_usd', 'salary_max_usd', 'remote_allowed', 'posted_date']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   job_id            500 non-null    int64 
 1   job_title         500 non-null    object
 2   company           500 non-null    object
 3   location          500 non-null    object
 4   employment_type   500 non-null    object
 5   experience_level  500 non-null    object
 6   industry          500 non-null    object
 7   skills_required   500 non-null    object
 8   salary_min_usd    500 non-null    int64 
 9   salary_max_usd    500 non-null    int64 
 10  remote_allowed    500 non-null    bool  
 11  posted_date       500 non-null    object
dtypes: bool(1),

In [28]:
# Cell 27: Missing Value Check

missing_values = df_linkedin.isnull().sum()

print("Missing values:")
display(missing_values)

Missing values:


job_id              0
job_title           0
company             0
location            0
employment_type     0
experience_level    0
industry            0
skills_required     0
salary_min_usd      0
salary_max_usd      0
remote_allowed      0
posted_date         0
dtype: int64

In [29]:
# Cell 28: Duplicate Check

print("Duplicate rows:",
      df_linkedin.duplicated().sum())

print("Duplicate Job IDs:",
      df_linkedin["job_id"].duplicated().sum())

Duplicate rows: 0
Duplicate Job IDs: 0


In [30]:
# Cell 29: Standardize Column Names

df_linkedin.columns = (
    df_linkedin.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df_linkedin.columns.tolist())

['job_id', 'job_title', 'company', 'location', 'employment_type', 'experience_level', 'industry', 'skills_required', 'salary_min_usd', 'salary_max_usd', 'remote_allowed', 'posted_date']


In [31]:
# Cell 30: Clean Text Columns

text_columns = [
    "job_title",
    "company",
    "location",
    "employment_type",
    "experience_level",
    "industry",
    "skills_required"
]

for col in text_columns:
    df_linkedin[col] = (
        df_linkedin[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Text cleaning completed.")

Text cleaning completed.


In [32]:
# Cell 31: Convert Salary Columns

df_linkedin["salary_min_usd"] = pd.to_numeric(
    df_linkedin["salary_min_usd"],
    errors="coerce"
)

df_linkedin["salary_max_usd"] = pd.to_numeric(
    df_linkedin["salary_max_usd"],
    errors="coerce"
)

print(df_linkedin[
    ["salary_min_usd", "salary_max_usd"]
].describe())

       salary_min_usd  salary_max_usd
count      500.000000      500.000000
mean     85446.254000   130056.608000
std      19961.595959    29015.206301
min      50540.000000    62652.000000
25%      68330.500000   106633.750000
50%      85917.000000   130570.000000
75%     102661.000000   151759.750000
max     119984.000000   196603.000000


In [33]:
# Cell 32: Salary Range Validation

print("Minimum salary:", df_linkedin["salary_min_usd"].min())
print("Maximum salary:", df_linkedin["salary_max_usd"].max())

print("\nInvalid minimum salaries:")
print(
    (df_linkedin["salary_min_usd"] < 10000).sum(),
    "records below $10,000"
)

print(
    (df_linkedin["salary_min_usd"] > 500000).sum(),
    "records above $500,000"
)

print("\nInvalid maximum salaries:")
print(
    (df_linkedin["salary_max_usd"] < 10000).sum(),
    "records below $10,000"
)

print(
    (df_linkedin["salary_max_usd"] > 500000).sum(),
    "records above $500,000"
)

Minimum salary: 50540
Maximum salary: 196603

Invalid minimum salaries:
0 records below $10,000
0 records above $500,000

Invalid maximum salaries:
0 records below $10,000
0 records above $500,000


In [34]:
# Cell 33: Remove Salary Outliers

before_rows = len(df_linkedin)

df_linkedin = df_linkedin[
    (df_linkedin["salary_min_usd"] >= 10000) &
    (df_linkedin["salary_min_usd"] <= 500000) &
    (df_linkedin["salary_max_usd"] >= 10000) &
    (df_linkedin["salary_max_usd"] <= 500000)
].copy()

after_rows = len(df_linkedin)

print("Rows before filtering:", before_rows)
print("Rows after filtering:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before filtering: 500
Rows after filtering: 500
Rows removed: 0


In [35]:
# Cell 34: Calculate Average Annual Salary

df_linkedin["salary_avg_usd"] = (
    df_linkedin["salary_min_usd"] +
    df_linkedin["salary_max_usd"]
) / 2

display(
    df_linkedin[
        [
            "job_title",
            "salary_min_usd",
            "salary_max_usd",
            "salary_avg_usd"
        ]
    ].head()
)

,job_title,salary_min_usd,salary_max_usd,salary_avg_usd
0,Product Manager,53278,99326,76302.0
1,Product Manager,76062,141049,108555.5
2,Machine Learning Engineer,62676,119728,91202.0
3,Data Analyst,75203,94319,84761.0
4,Product Manager,98520,155086,126803.0


In [36]:
# Cell 35: Convert USD Salary to Indian LPA

USD_TO_INR = 83.0

df_linkedin["salary_min_inr"] = (
    df_linkedin["salary_min_usd"] * USD_TO_INR
)

df_linkedin["salary_max_inr"] = (
    df_linkedin["salary_max_usd"] * USD_TO_INR
)

df_linkedin["salary_avg_inr"] = (
    df_linkedin["salary_avg_usd"] * USD_TO_INR
)

# Convert INR to LPA
# 1 Lakh = 100,000 INR

df_linkedin["salary_min_lpa"] = (
    df_linkedin["salary_min_inr"] / 100000
)

df_linkedin["salary_max_lpa"] = (
    df_linkedin["salary_max_inr"] / 100000
)

df_linkedin["salary_avg_lpa"] = (
    df_linkedin["salary_avg_inr"] / 100000
)

display(
    df_linkedin[
        [
            "job_title",
            "salary_avg_usd",
            "salary_avg_lpa"
        ]
    ].head()
)


,job_title,salary_avg_usd,salary_avg_lpa
0,Product Manager,76302.0,63.330660
1,Product Manager,108555.5,90.101065
2,Machine Learning Engineer,91202.0,75.697660
3,Data Analyst,84761.0,70.351630
4,Product Manager,126803.0,105.246490


In [37]:
# Cell 36: Parse Required Skills

def parse_skills(value):
    if pd.isna(value):
        return []
    
    skills = str(value).split(",")
    
    return [
        skill.strip().lower()
        for skill in skills
        if skill.strip()
    ]


df_linkedin["skills_list"] = (
    df_linkedin["skills_required"]
    .apply(parse_skills)
)

display(
    df_linkedin[
        ["job_title", "skills_required", "skills_list"]
    ].head()
)

,job_title,skills_required,skills_list
0,Product Manager,"Python, Node.js, SQL, JavaScript, Power BI, Ma...","[python, node.js, sql, javascript, power bi, m..."
1,Product Manager,"AWS, Java, JavaScript, Azure, SQL","[aws, java, javascript, azure, sql]"
2,Machine Learning Engineer,"Node.js, Docker, SQL","[node.js, docker, sql]"
3,Data Analyst,"Docker, AWS, Kubernetes","[docker, aws, kubernetes]"
4,Product Manager,"Java, Kubernetes, Docker, AWS","[java, kubernetes, docker, aws]"


In [38]:
# Cell 37: Skill Frequency

skill_counter = Counter()

for skills in df_linkedin["skills_list"]:
    for skill in set(skills):
        skill_counter[skill] += 1

skill_frequency_linkedin = pd.DataFrame(
    skill_counter.items(),
    columns=["Skill", "Job_Count"]
)

skill_frequency_linkedin = (
    skill_frequency_linkedin
    .sort_values(
        by="Job_Count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    skill_frequency_linkedin.head(20)
)

,Skill,Job_Count
0,excel,165
1,azure,161
2,aws,161
3,node.js,158
4,power bi,157
5,tableau,151
6,sql,149
7,kubernetes,149
8,react,147
9,machine learning,147


In [39]:
# Cell 38: Skill Demand Ratio

total_jobs = len(df_linkedin)

skill_frequency_linkedin["Demand_Ratio"] = (
    skill_frequency_linkedin["Job_Count"] /
    total_jobs
)

skill_frequency_linkedin["Demand_Percentage"] = (
    skill_frequency_linkedin["Demand_Ratio"] * 100
)

display(
    skill_frequency_linkedin.head(20)
)

,Skill,Job_Count,Demand_Ratio,Demand_Percentage
0,excel,165,0.330,33.0
1,azure,161,0.322,32.2
2,aws,161,0.322,32.2
3,node.js,158,0.316,31.6
4,power bi,157,0.314,31.4
5,tableau,151,0.302,30.2
6,sql,149,0.298,29.8
7,kubernetes,149,0.298,29.8
8,react,147,0.294,29.4
9,machine learning,147,0.294,29.4


In [40]:
# Cell 39: Keyword-Family Inverted Index

keyword_families = {
    "software_engineer": [
        "software engineer",
        "software developer",
        "application developer",
        "backend engineer",
        "backend developer",
        "full stack developer",
        "full-stack developer"
    ],
    
    "machine_learning_engineer": [
        "machine learning engineer",
        "ml engineer",
        "machine learning developer",
        "ml developer",
        "ai engineer",
        "artificial intelligence engineer"
    ],
    
    "data_scientist": [
        "data scientist",
        "data science",
        "data analyst",
        "analytics"
    ],
    
    "frontend_developer": [
        "frontend developer",
        "front end developer",
        "ui developer",
        "web developer"
    ],
    
    "backend_developer": [
        "backend developer",
        "back end developer",
        "backend engineer"
    ]
}


linkedin_inverted_index = defaultdict(set)

for idx, title in df_linkedin["job_title"].items():
    
    title_lower = title.lower()
    
    for family, keywords in keyword_families.items():
        
        for keyword in keywords:
            
            if keyword in title_lower:
                linkedin_inverted_index[family].add(idx)


for family, indices in linkedin_inverted_index.items():
    print(
        family,
        "->",
        len(indices),
        "jobs"
    )

machine_learning_engineer -> 46 jobs
data_scientist -> 106 jobs
software_engineer -> 100 jobs


In [41]:
# Cell 40: Search Jobs by Role

def search_linkedin_jobs(role):
    
    role = role.lower().strip()
    
    if role not in linkedin_inverted_index:
        return pd.DataFrame()
    
    indices = list(
        linkedin_inverted_index[role]
    )
    
    return df_linkedin.loc[
        indices
    ].reset_index(drop=True)


# Example
ml_jobs = search_linkedin_jobs(
    "machine_learning_engineer"
)

display(ml_jobs.head(10))

,job_id,job_title,company,location,employment_type,experience_level,industry,skills_required,salary_min_usd,salary_max_usd,remote_allowed,posted_date,salary_avg_usd,salary_min_inr,salary_max_inr,salary_avg_inr,salary_min_lpa,salary_max_lpa,salary_avg_lpa,skills_list
0,3,Machine Learning Engineer,Salesforce,"London, UK",Full-time,Director,Education,"Node.js, Docker, SQL",62676,119728,False,25-01-2024,91202.0,5202108.0,9937424.0,7569766.0,52.02108,99.37424,75.697660,"[node.js, docker, sql]"
1,383,Machine Learning Engineer,Amazon,"Toronto, Canada",Internship,Director,Finance,"JavaScript, Node.js, Python, Docker",115840,187281,True,18-05-2023,151560.5,9614720.0,15544323.0,12579521.5,96.14720,155.44323,125.795215,"[javascript, node.js, python, docker]"
2,394,Machine Learning Engineer,IBM,"San Francisco, USA",Internship,Entry level,Healthcare,"Java, SQL, Power BI, AWS, Azure, Machine Learning",89229,141541,False,06-04-2023,115385.0,7406007.0,11747903.0,9576955.0,74.06007,117.47903,95.769550,"[java, sql, power bi, aws, azure, machine lear..."
3,398,Machine Learning Engineer,IBM,"New York, USA",Part-time,Mid level,Healthcare,"Node.js, Excel, Kubernetes, Tableau, Python",64920,118604,True,07-01-2023,91762.0,5388360.0,9844132.0,7616246.0,53.88360,98.44132,76.162460,"[node.js, excel, kubernetes, tableau, python]"
4,149,Machine Learning Engineer,Netflix,"Seattle, USA",Part-time,Director,E-commerce,"SQL, Docker, Excel, Node.js, Java",74405,86251,True,21-09-2023,80328.0,6175615.0,7158833.0,6667224.0,61.75615,71.58833,66.672240,"[sql, docker, excel, node.js, java]"
5,23,Machine Learning Engineer,Meta,"London, UK",Internship,Senior level,Technology,"Excel, Docker, AWS",86930,124537,True,06-10-2023,105733.5,7215190.0,10336571.0,8775880.5,72.15190,103.36571,87.758805,"[excel, docker, aws]"
6,280,Machine Learning Engineer,Adobe,"Bangalore, India",Contract,Director,Technology,"Tableau, Excel, JavaScript",79218,148784,True,25-02-2023,114001.0,6575094.0,12349072.0,9462083.0,65.75094,123.49072,94.620830,"[tableau, excel, javascript]"
7,153,Machine Learning Engineer,Microsoft,"San Francisco, USA",Full-time,Mid level,Education,"Java, Azure, Machine Learning, Kubernetes",108556,187784,True,22-06-2024,148170.0,9010148.0,15586072.0,12298110.0,90.10148,155.86072,122.981100,"[java, azure, machine learning, kubernetes]"
8,412,Machine Learning Engineer,Meta,"Toronto, Canada",Internship,Senior level,Education,"Java, Excel, React, SQL",100307,118965,False,08-05-2024,109636.0,8325481.0,9874095.0,9099788.0,83.25481,98.74095,90.997880,"[java, excel, react, sql]"
9,29,Machine Learning Engineer,FinTechHub,"Berlin, Germany",Internship,Entry level,E-commerce,"AWS, Python, Power BI, Docker",64208,131167,True,09-02-2023,97687.5,5329264.0,10886861.0,8108062.5,53.29264,108.86861,81.080625,"[aws, python, power bi, docker]"


In [42]:
# Cell 41: Salary Summary by Job Title

salary_summary = (
    df_linkedin
    .groupby("job_title")
    .agg(
        Job_Count=("job_id", "count"),
        Average_Salary_LPA=("salary_avg_lpa", "mean"),
        Minimum_Salary_LPA=("salary_min_lpa", "mean"),
        Maximum_Salary_LPA=("salary_max_lpa", "mean")
    )
    .sort_values(
        by="Average_Salary_LPA",
        ascending=False
    )
    .reset_index()
)

display(salary_summary.head(20))

,job_title,Job_Count,Average_Salary_LPA,Minimum_Salary_LPA,Maximum_Salary_LPA
0,DevOps Engineer,49,92.183171,73.735286,110.631056
1,Machine Learning Engineer,46,91.748795,73.015677,110.481913
2,Data Analyst,59,90.162766,70.254239,110.071294
3,Full Stack Developer,52,89.986270,71.133650,108.838890
4,Product Manager,47,89.513346,72.362543,106.664148
5,AI Researcher,47,89.383362,71.977247,106.789478
6,Software Engineer,48,88.689079,69.782025,107.596134
7,Marketing Analyst,57,88.525943,69.776236,107.275651
8,Business Analyst,48,88.050887,69.637259,106.464515
9,Data Scientist,47,86.018984,67.896896,104.141071


In [43]:
# Cell 42: Create Final Job Records

final_jobs = []

for _, row in df_linkedin.iterrows():
    
    record = {
        "job_id": int(row["job_id"]),
        "job_title": row["job_title"],
        "company": row["company"],
        "location": row["location"],
        "employment_type": row["employment_type"],
        "experience_level": row["experience_level"],
        "industry": row["industry"],
        "skills": row["skills_list"],
        "salary_min_usd": float(row["salary_min_usd"]),
        "salary_max_usd": float(row["salary_max_usd"]),
        "salary_avg_usd": float(row["salary_avg_usd"]),
        "salary_min_lpa": float(row["salary_min_lpa"]),
        "salary_max_lpa": float(row["salary_max_lpa"]),
        "salary_avg_lpa": float(row["salary_avg_lpa"]),
        "remote_allowed": row["remote_allowed"],
        "posted_date": row["posted_date"]
    }
    
    final_jobs.append(record)


print("Final job records:", len(final_jobs))

Final job records: 500


In [44]:
# Cell 43: Save LinkedIn Job Data as JSON

with open(
    "linkedin_jobs_processed.json",
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        final_jobs,
        f,
        indent=2,
        ensure_ascii=False
    )

print("linkedin_jobs_processed.json saved successfully!")

linkedin_jobs_processed.json saved successfully!


In [45]:
# Cell 44: Save Processed LinkedIn Dataset

df_linkedin.to_csv(
    "linkedin_jobs_processed.csv",
    index=False
)

print("linkedin_jobs_processed.csv saved successfully!")


linkedin_jobs_processed.csv saved successfully!


In [46]:
# Cell 45: Final Validation

print("========== JOB MARKET FINAL VALIDATION ==========")

print("Total records:", len(df_linkedin))

print(
    "Unique Job IDs:",
    df_linkedin["job_id"].nunique()
)

print(
    "Missing salary averages:",
    df_linkedin["salary_avg_usd"].isna().sum()
)

print(
    "Minimum average salary (USD):",
    df_linkedin["salary_avg_usd"].min()
)

print(
    "Maximum average salary (USD):",
    df_linkedin["salary_avg_usd"].max()
)

print(
    "Minimum average salary (LPA):",
    df_linkedin["salary_avg_lpa"].min()
)

print(
    "Maximum average salary (LPA):",
    df_linkedin["salary_avg_lpa"].max()
)

print("\nTop 10 demanded skills:")

display(
    skill_frequency_linkedin.head(10)
)

print("\nFiles created:")
print("1. linkedin_jobs_processed.json")
print("2. linkedin_jobs_processed.csv")

print("\nJob Market LinkedIn processing completed!")

========== JOB MARKET FINAL VALIDATION ==========
Total records: 500
Unique Job IDs: 500
Missing salary averages: 0
Minimum average salary (USD): 56877.5
Maximum average salary (USD): 157321.0
Minimum average salary (LPA): 47.208325
Maximum average salary (LPA): 130.57643

Top 10 demanded skills:


,Skill,Job_Count,Demand_Ratio,Demand_Percentage
0,excel,165,0.330,33.0
1,azure,161,0.322,32.2
2,aws,161,0.322,32.2
3,node.js,158,0.316,31.6
4,power bi,157,0.314,31.4
5,tableau,151,0.302,30.2
6,sql,149,0.298,29.8
7,kubernetes,149,0.298,29.8
8,react,147,0.294,29.4
9,machine learning,147,0.294,29.4



Files created:
1. linkedin_jobs_processed.json
2. linkedin_jobs_processed.csv

Job Market LinkedIn processing completed!


In [47]:
# Combine both processed datasets into one Job Market dataset

combined_jobs = pd.concat(
    [
        df_jobs,
        df_linkedin
    ],
    ignore_index=True,
    sort=False
)

print("Total combined records:", len(combined_jobs))
print("Total columns:", len(combined_jobs.columns))

Total combined records: 700
Total columns: 28


In [48]:
# Save combined Job Market dataset

combined_jobs.to_csv(
    "jobs_combined_700.csv",
    index=False
)

print("Saved successfully as jobs_combined_700.csv")

Saved successfully as jobs_combined_700.csv


In [49]:
# Convert combined dataset to JSON

combined_jobs.to_json(
    "jobs.json",
    orient="records",
    indent=2
)

print("Saved successfully as jobs.json")

Saved successfully as jobs.json


In [50]:
print("Final Job Market Dataset")
print("========================")
print("Records:", len(combined_jobs))
print("Expected:", 700)

display(combined_jobs.head())

Final Job Market Dataset
Records: 700
Expected: 700


,job_id,job_title,industry,required_education,required_degree_field,years_of_experience,hard_skills,soft_skills,hard_skills_normalized,soft_skills_normalized,...,remote_allowed,posted_date,salary_avg_usd,salary_min_inr,salary_max_inr,salary_avg_inr,salary_min_lpa,salary_max_lpa,salary_avg_lpa,skills_list
0,1,Software Engineer,Technology,Bachelor,Computing,2.0,"[Python, Java, C++, Git, Version Control, Algo...","[Problem-Solving, Teamwork]","[python, java, c++, git, version control, algo...","[problem-solving, teamwork]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Frontend Developer,Technology,Bachelor,Computing,1.0,"[HTML, CSS, JavaScript, React, TypeScript, Res...","[Creativity, Attention to Detail]","[html, css, javascript, react, typescript, res...","[creativity, attention to detail]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Backend Developer,Technology,Bachelor,Computing,2.0,"[Java, Python, Node.js, Spring Boot, REST APIs...","[Logical Thinking, Collaboration]","[java, python, node.js, spring boot, rest apis...","[logical thinking, collaboration]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Full Stack Developer,Technology,Bachelor,Computing,3.0,"[JavaScript, React, Node.js, MongoDB, Express....","[Adaptability, Communication]","[javascript, react, node.js, mongodb, express....","[adaptability, communication]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,DevOps Engineer,Technology,Bachelor,Computing,3.0,"[Docker, Kubernetes, CI/CD, AWS, Terraform, Je...","[Problem-Solving, Teamwork]","[docker, kubernetes, ci/cd, aws, terraform, je...","[problem-solving, teamwork]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
